In [1]:
from topic_gen.generate import Generator
import ir_datasets
import pandas as pd
import os
from topic_gen.models import TRECTopic, Topics
from langchain.chat_models import init_chat_model

from topic_gen import logger
logger.setLevel("INFO")

In [2]:
class uqv_parser:
    def __init__(self):
        self.queries = []
        self.dataset = ir_datasets.load("disks45/nocr/trec-robust-2004")
        self.store = self.dataset.docs_store()

        self.qrels_map = self.make_qrels_map()

    def make_qrels_map(self):
        qrels_map = {}
        for qrel in self.dataset.qrels_iter():
            if not qrels_map.get(qrel.query_id):
                qrels_map[qrel.query_id] = []
            if qrel.relevance > 0:
                doc = self.store.get(qrel.doc_id)
                qrels_map[qrel.query_id].append(doc.body)
        return qrels_map

    def parse_variants(self):
        # variants
        uqv = pd.read_csv(
            "../experiments/data/raw/robust-uqv.txt", sep=";", names=["query_id", "uqv"]
        )
        uqv["qid"] = uqv["query_id"].apply(lambda x: x.split("-")[0])

        for query in self.dataset.queries_iter():
            variants = uqv[uqv["qid"] == query.query_id]["uqv"].to_list()

            self.queries.append(
                {
                    "qid": query.query_id,
                    "title": query.title,
                    "description": query.description,
                    "narrative": query.narrative,
                    "uqv": variants,
                    "rel_docs": self.qrels_map.get(query.query_id),
                }
            )

        return self.queries

In [6]:
def generate_topics_robust_uqv(
    topics,
    model = "llama3.3:70b",
    prompt_name = "trec-base",
    dataset_name = "robust04",
    n_queries = 3,
    n_docs = 1,
    output=".",
):
    # Generator
    llm = init_chat_model(
        model=model,
        temperature=0.0,
        base_url="http://139.6.160.39:6543/v1",
        api_key="not-needed",
        model_provider="openai", 
        )
    generator = Generator(llm=llm, output_class=TRECTopic)


    for topic in topics:
        generated_topic = generator.generate(
            prompt_name=prompt_name,
            number_of_topics=1,
            queries="\n".join(topic["uqv"][:n_queries]),
            clicked_documents="\n\n".join(topic["rel_docs"][:n_docs]),
        )
        
        topic_dict = generated_topic.model_dump()
        topic_dict["qid"] = topic["qid"]
        
        # Save
        with open(os.path.join(output, f"topics-{dataset_name}-{model}-{prompt_name}-q{n_queries}-d{n_docs}.jsonl"), "a+") as f:
            f.write(f"{topic_dict}\n")

In [13]:
llm = init_chat_model(
        model="Satwik11/Llama-3.3-70B-Instruct-AutoRound-GPTQ-4bit",
        temperature=0.0,
        base_url="http://139.6.160.39:6543/v1",
        api_key="not-needed",
        model_provider="openai", 
        )

In [14]:
llm.invoke("Hello world")  # test connection

APIConnectionError: Connection error.

In [7]:
parser = uqv_parser()
topics = parser.parse_variants()

In [9]:
generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=1,
    n_docs=1,
    )

[topic_gen] [INFO] Final text prompt:
 Objective: Generate a high-quality, structured TREC-style topic that simulates a nuanced user information need. This topic will be synthesized from a given set of user search queries and a collection of text snippets from documents the user has identified as relevant.
Instructions:
You are an expert in Information Science and a seasoned analyst for the Text REtrieval Conference (TREC). Your task is to create a formal TREC topic that represents the underlying information need of a user. You will be provided with two pieces of evidence: a list of queries the user has searched for, and a set of excerpts from documents they have judged as relevant.
Your goal is to move beyond the specific keywords in the queries and the explicit text in the documents to define the users broader, more abstract information goal. You must articulate what makes a document truly relevant to this users need, including the specific criteria for inclusion and exclusion.

Cont

AttributeError: 'NoneType' object has no attribute 'model_dump'

In [ ]:
generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=2,
    n_docs=1,
    )

generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=3,
    n_docs=1,
    )
generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=4,
    n_docs=1,
    )
generate_topics_robust_uqv(
    topics=topics[200:210], 
    output="../experiments/data/interim/",
    n_queries=5,
    n_docs=1,
    )

[topic_gen] [INFO] Final text prompt:
 Objective:
To generate a high-quality, structured TREC-style topic that simulates a nuanced user information need. This topic will be synthesized from a given set of user search queries and a collection of text snippets from documents the user has identified as relevant.
Instructions:
You are an expert in Information Science and a seasoned analyst for the Text REtrieval Conference (TREC). Your task is to create a formal TREC topic that represents the underlying information need of a user. You will be provided with two pieces of evidence: a list of queries the user has searched for, and a set of excerpts from documents they have judged as relevant.
Your goal is to move beyond the specific keywords in the queries and the explicit text in the documents to define the users broader, more abstract information goal. You must articulate what makes a document truly relevant to this users need, including the specific criteria for inclusion and exclusion.

C